# T06: Schema Roundtrip Validation

This tutorial demonstrates offline roundtrip validation of custom schemas using
the `undata` ingestion library and CLI. No backend or other services are required.

**Services required**: NONE (fully offline)

**Requires**: undata library (`cd ../ingestion && uv sync`)

**Est. time**: 3 min

In [1]:
# Cell 2 — path setup only (NO service skip cell — fully offline)
import os
import subprocess
from pathlib import Path

INGESTION_DIR = Path(
    os.getenv("INGESTION_DIR", str(Path("../ingestion").resolve()))
).resolve()
FIXTURES = INGESTION_DIR / "tests" / "fixtures"

assert FIXTURES.exists(), (
    f"Fixtures not found at {FIXTURES} — run `cd ../ingestion && uv sync` first"
)
print(f"✓ Using fixtures from {FIXTURES}")
print(f"  JSON sample:   {(FIXTURES / 'generic_schema_sample.json').exists()}")
print(f"  LinkML sample: {(FIXTURES / 'linkml_sample.yaml').exists()}")

✓ Using fixtures from /Users/satra/software/undata/ingestion/tests/fixtures
  JSON sample:   True
  LinkML sample: True


## 1. Import a JSON Schema

`GenericJSONSchemaAdapter` handles JSON Schema draft-07, 2019-09, and 2020-12.
It extracts normalized elements from the schema's properties and `$defs`.

In [2]:
result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        f"""
from undata.adapters.json_schema import GenericJSONSchemaAdapter
adapter = GenericJSONSchemaAdapter()
adapter.load_file({str(FIXTURES / 'generic_schema_sample.json')!r})
elements = adapter.extract_elements()
classes = adapter.extract_classes()
print(f'Extracted {{len(elements)}} elements, {{len(classes)}} classes')
for el in elements:
    print(f'  - {{el.name}} | data_type={{el.data_type}} | required={{el.required}}')
""",
    ],
    cwd=str(INGESTION_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
assert result.returncode == 0, f"Failed with code {result.returncode}"

Extracted 7 elements, 2 classes
  - id | data_type=string | required=True
  - count | data_type=number | required=False
  - active | data_type=boolean | required=False
  - home_address | data_type=object | required=False
  - status | data_type=string | required=False
  - street | data_type=string | required=False
  - city | data_type=string | required=False



## 2. Roundtrip: JSON Schema → LinkML → Re-import

`roundtrip_json_schema()` converts a JSON Schema to LinkML, serializes it, then
re-imports it and compares element sets. A `fidelity_score` of 1.0 means all
elements survived the roundtrip without loss.

In [3]:
result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        f"""
from undata.roundtrip import roundtrip_json_schema
result = roundtrip_json_schema({str(FIXTURES / 'generic_schema_sample.json')!r})
print(f'Fidelity score:   {{result.fidelity_score:.2f}}')
print(f'Missing elements: {{result.missing_elements}}')
print(f'Missing classes:  {{result.missing_classes}}')
if result.warnings:
    print(f'Warnings:         {{result.warnings}}')
assert result.fidelity_score == 1.0, f'Expected 1.0, got {{result.fidelity_score}}'
print('PASS: fidelity = 1.0')
""",
    ],
    cwd=str(INGESTION_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
assert result.returncode == 0, f"Roundtrip failed with code {result.returncode}"
assert "PASS" in result.stdout, "Expected PASS in output"
print("✓ JSON Schema roundtrip: fidelity = 1.0")

Fidelity score:   1.00
Missing elements: []
Missing classes:  []
PASS: fidelity = 1.0

✓ JSON Schema roundtrip: fidelity = 1.0


## 3. Import a LinkML YAML Schema

`LinkMLAdapter` loads LinkML YAML schemas using `linkml_runtime.yaml_loader`.

In [4]:
result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        f"""
from undata.adapters.linkml_adapter import LinkMLAdapter
la = LinkMLAdapter()
la.load_file({str(FIXTURES / 'linkml_sample.yaml')!r})
elements = la.extract_elements()
classes = la.extract_classes()
print(f'Extracted {{len(elements)}} elements, {{len(classes)}} classes')
print('Slots:')
for el in elements:
    print(f'  - {{el.name}} | data_type={{el.data_type}} | source_local_id={{el.source_local_id}}')
""",
    ],
    cwd=str(INGESTION_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
assert result.returncode == 0, f"Failed with code {result.returncode}"

Extracted 4 elements, 2 classes
Slots:
  - name | data_type=string | source_local_id=test_schema.name
  - age | data_type=number | source_local_id=test_schema.age
  - active | data_type=boolean | source_local_id=test_schema.active
  - tags | data_type=array | source_local_id=test_schema.tags



## 4. Roundtrip: LinkML → Re-serialize → Re-import

`roundtrip_linkml()` serializes the schema back to YAML and re-imports it.
We expect a high fidelity score (≥ 0.8) since LinkML is the native format.

In [5]:
result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        f"""
from undata.roundtrip import roundtrip_linkml
result = roundtrip_linkml({str(FIXTURES / 'linkml_sample.yaml')!r})
print(f'Fidelity score:   {{result.fidelity_score:.2f}}')
print(f'Missing elements: {{result.missing_elements}}')
print(f'Missing classes:  {{result.missing_classes}}')
assert result.fidelity_score >= 0.8, f'Expected >= 0.8, got {{result.fidelity_score}}'
print(f'PASS: fidelity = {{result.fidelity_score:.2f}}')
""",
    ],
    cwd=str(INGESTION_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
assert result.returncode == 0, f"Roundtrip failed with code {result.returncode}"
assert "PASS" in result.stdout
print("✓ LinkML roundtrip passed")

Fidelity score:   1.00
Missing elements: []
Missing classes:  []
PASS: fidelity = 1.00

✓ LinkML roundtrip passed


## 5. Using the CLI

The `undata roundtrip` CLI command performs the same roundtrip validation from the
command line. Exit code 0 = fidelity ≥ 1.0; exit code 1 = below threshold.

In [6]:
result = subprocess.run(
    [
        "uv",
        "run",
        "undata",
        "roundtrip",
        str(FIXTURES / "generic_schema_sample.json"),
    ],
    cwd=str(INGESTION_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
assert result.returncode == 0, f"roundtrip CLI failed with code {result.returncode}"
print("✓ undata roundtrip CLI: exit code 0")

Roundtrip fidelity: 1.00 (PASS)
  Missing elements:  0
  Missing classes:   0

STDERR: warning: `VIRTUAL_ENV=/Users/satra/software/undata/tutorials/.venv` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead
/Users/satra/software/undata/ingestion/.venv/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(

✓ undata roundtrip CLI: exit code 0


## Next Steps

You've validated JSON Schema and LinkML roundtrips entirely offline — no services needed.

This tutorial required no running services — all processing was offline.

Next: **[T07: Data Migration](07_data_migration.ipynb)** — diff schema sources,
create migration pathways, and run batch migrations (requires backend + migration-api).